# 1. Merge & Pre-processing
The cleaned transactions (`train`) and the cleaned customer table (`clientes`) are merged into one dataset, split into training and validation folds, and pre-processed in order: outliers, missing values, and encoding and scaling.

In [13]:
import warnings
from collections import namedtuple
from functools import partial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (needed for IterativeImputer)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler

# Same seed everywhere, so every run gives the same splits and draws
RANDOM_STATE = 42

____________________________________________________________________________________________________

## 1.1. Data Loading

In [14]:
trainFile = "../../data/trainDataset/train_2_4_1.csv"
clientFile = "../../data/clientesDataset/clientes_clean.csv"
testFile = "../../data/test.csv"

train = pd.read_csv(trainFile)
clientes = pd.read_csv(clientFile)
test = pd.read_csv(testFile)

RANDOM_STATE= 42

____________________________________________________________________________________________________

## 1.2. Merging the Datasets

In [15]:
train_clientes = train.merge(clientes, on="customer_id", how="left", validate="many_to_one")
assert len(train_clientes) == len(train)

train_clientes.head()

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,island,municipality,city,postal_area,postal_code,street,address,company_name,customer_since_year,since_2015_or_earlier
0,558.11,510.90,453.75,104.36,0.0,0.0,57.15,153.099911,4.0,1.0,...,(mainland),Municipality MU89: Council of Last Minute Edits,City CY286: Cached Result Falls,Postal area PA228: Hidden Sheet Sector,Postal code PC444: #REF! Delivery Route,Street ST303: Absolute Reference Terrace,Address AD761: Cell Without a Home,Excel Is Not A Database Ltd 1013,2024,0
1,978.80,830.02,795.77,183.03,0.0,0.0,34.25,472.943134,32.0,1.0,...,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,City CY147: VLOOKUP Bay,Postal area PA260: Hidden Sheet Sector,Postal code PC636: #REF! Delivery Route,NaN,Address AD382: Cell Without a Home,Excel Is Not A Database Ltd 35,2015,1
2,10.33,12.00,8.40,1.93,0.0,0.0,3.60,6.097485,1.0,1.0,...,(mainland),Municipality MU70: Council of Last Minute Edits,City CY265: Where-less Query Falls,Postal area PA84: Hidden Sheet Sector,Postal code PC200: #REF! Delivery Route,NaN,Address AD723: Cell Without a Home,Excel Is Not A Database Ltd 826,2017,0
3,856.83,696.61,696.61,160.22,0.0,0.0,0.00,408.167275,9.0,1.0,...,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,NaN,Postal area PA253: Hidden Sheet Sector,Postal code PC558: #REF! Delivery Route,Street ST144: Last Save Court,Address AD399: Cell Without a Home,Excel Is Not A Database Ltd 530,2018,0
4,1616.10,1488.85,1313.90,302.20,0.0,0.0,174.95,485.401864,33.0,1.0,...,(mainland),Municipality MU142: Council of Last Minute Edits,City CY359: Spill Error Ridge,Postal area PA136: Hidden Sheet Sector,Postal code PC300: #REF! Delivery Route,Street ST395: AutoFill Path,Address AD928: Cell Without a Home,Excel Is Not A Database Ltd 220,2017,0


**Conclusions:**
* **Lossless merge:** every one of the **6,469** transactions found its customer (`validate="many_to_one"` confirms one row per `customer_id` in `clientes`), so no rows were lost or duplicated. The merged table has **57 columns**: the 45 of `train` plus 12 customer attributes.
* **Customer coverage:** the transactions belong to **607** customers, so 536 of the 1,143 customers in `clientes` have no purchase in `train` and are simply not used.
* **New missing values:** the customer attributes bring their own gaps into the merged table, mostly in identifier-like columns: `street` (58.9%), `city` (32.9%) and `postal_code` (3.4%). The descriptive ones are almost complete: `postal_area` (1.0%), `commercial_region` (0.5%), `island` and `municipality` (0.1%). These are handled together with the `train` gaps in the next sections.

____________________________________________________________________________________________________

## 1.3. Feature Type Separation
Imputation and encoding work differently on numbers and on labels (a median or a KNN fill needs numbers, a mode or an `"Unknown"` level works on labels), so the merged columns are split into groups first:

1. **Target:** `repeat_purchase_90d`, the variable to predict.
2. **Identifiers:** `ID` (one per row) and `customer_id` (the join key, kept to group the cross-validation folds by customer).
3. **Excluded:** `company_name`, `address`, `street` and `postal_code` are distinct for almost every customer, so they name a customer instead of describing one; `random_noise` carries no information by construction.
4. **Numeric** and **categorical** features: everything else, separated by data type.

In [ ]:
target = "repeat_purchase_90d"
identifiers = ["ID", "customer_id"]

# Unique per customer (they name a customer instead of describing one) or pure noise
excluded = ["company_name", "address", "street", "postal_code", "random_noise"]

features = [c for c in train_clientes.columns if c not in [target] + identifiers + excluded]

# Automatic separation between numerical and categorical variables
numeric = [c for c in features if pd.api.types.is_numeric_dtype(train_clientes[c])]
categorical = [c for c in features if c not in numeric]

# Every column belongs to exactly one group
assert set([target] + identifiers + excluded + numeric + categorical) == set(train_clientes.columns)

features = train_clientes[numeric + categorical]
identifier = train_clientes[target]

print(f"{len(numeric)} numeric | {len(categorical)} categorical | {len(identifiers)} identifiers | {len(excluded)} excluded")
print("categorical:", categorical)

37 numeric | 12 categorical | 2 identifiers | 5 excluded
categorical: ['payment_type', 'salesperson', 'commercial_zone', 'transport_method', 'warehouse', 'document_series', 'country_or_region', 'commercial_region', 'island', 'municipality', 'city', 'postal_area']


**Conclusions:**
* **49 features** are left for the model: **37 numeric** and **12 categorical**. The target has no missing values and is fairly balanced (56% of purchases are repeated within 90 days).
* **Numbers that are labels:** some numeric columns are categories written as numbers: `purchase_month`, `purchase_dayofweek`, `purchase_quarter` and the 0/1 flags `is_first_purchase`, `is_weekend` and `since_2015_or_earlier`. None of them has missing values, so they do not affect the imputation; whether they are encoded as categories is decided in the encoding step.

____________________________________________________________________________________________________

## 1.4. Missing Values Overview

In this step, we measure the missing values, which gave us some information about which features still have gaps after the merge, how many, and whether if a value is missing says something about the target.

**Conclusions:**

____________________________________________________________________________________________________

## 1.5. Train / Validation Split

### 1.5.1. Evaluation Setup
Every comparison below runs through the same three tools, written once, as in the course practical: the **model matrix** (`design_matrix`), the **fill** that makes a split ready for a model (`fill_missing`), and the **scorer** (`held_out_f1`), which trains a logistic regression on the training half of each split and returns its F1 on the untouched validation half.

In [ ]:
# The table every comparison works on: the features plus the target
frame = features.join(identifier)

# Logistic regression may stop before converging on unscaled columns; scaling is compared in 1.8
warnings.filterwarnings("ignore", category=ConvergenceWarning)


def design_matrix(frame, numeric, categorical):
    """Numbers as they are, everything else one-hot. Identifiers never enter."""
    encoder = OneHotEncoder(sparse_output=False, dtype=float)
    encoded = encoder.fit_transform(frame[categorical].astype("string"))
    text = pd.DataFrame(
        encoded,
        columns=encoder.get_feature_names_out(categorical),
        index=frame.index,
    )
    return pd.concat([frame[numeric], text], axis=1)


def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs: each imputer is fitted on the
    training rows of its own columns and fills those columns in both halves, so
    the validation rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, so every gap becomes np.nan
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        if isinstance(imputer, (KNNImputer, IterativeImputer)):
            # Both are sensitive to scale, so they work on standardised columns
            # and their fills go back in the columns' own units
            scale = StandardScaler().fit(blocks[0])
            imputer.fit(scale.transform(blocks[0]))
            filled = [scale.inverse_transform(imputer.transform(scale.transform(block)))
                      for block in blocks]
        else:
            imputer.fit(blocks[0])
            filled = [imputer.transform(block) for block in blocks]
        train[columns], test[columns] = filled
    return train, test


# A model cannot fit a missing value, so every comparison fills inside its own split.
# Until section 1.7 chooses a plan, that fill is the training median for the numbers
# and the training mode for the labels.
median_mode = [(numeric, SimpleImputer(strategy="median")),
               (categorical, SimpleImputer(strategy="most_frequent"))]

Scored = namedtuple("Scored", "f1 iterations")


def sem(values):
    """Standard error of the mean: how far it would move on new splits."""
    return values.std(ddof=1) / np.sqrt(len(values))


def held_out_f1(prepare, frame, splits, numeric, categorical,
                target="repeat_purchase_90d", scaler=None):
    """Score one treatment by F1 on each split's untouched validation rows.

    `prepare(train, test)` applies the treatment, fitted on `train` only. The
    result holds the validation F1 per split (`.f1`) and the solver's iterations
    (`.iterations`). `scaler`, when given, is fitted on the training half and
    applied to both.
    """
    f1, iterations = [], []
    for train_rows, test_rows in splits:
        train, test = prepare(frame.iloc[train_rows], frame.iloc[test_rows])
        matrix = design_matrix(pd.concat([train, test]), numeric, categorical)
        a, b = matrix.iloc[:len(train)], matrix.iloc[len(train):]
        if scaler is not None:
            fitted = scaler.fit(a)
            a, b = fitted.transform(a), fitted.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[target])
        iterations.append(int(model.n_iter_[0]))
        f1.append(f1_score(test[target], model.predict(b)))
    return Scored(np.array(f1), np.array(iterations))

### 1.5.2. The Splits
Following the course practical (`week_03_deepen_exploration`), instead of a single train/validation split the data is divided **20 times** with `StratifiedShuffleSplit` (80% training, 20% validation, `random_state=42`), and every candidate treatment is scored on the **same 20 validation sets**. A difference between two treatments therefore comes from the treatments, not from a lucky split, and it can be judged by its mean F1 and by how many of the 20 splits it wins.

* **Stratified:** each split keeps the class balance of the target `repeat_purchase_90d`, so training and validation have the same share of repeat purchases.
* **Fitted on training rows only:** every step that learns from the data (outlier limits, imputers, encoders, scalers) is fitted on the 80% training rows of each split and only applied to its 20% validation rows. A median computed over validation rows would let the validation set leak into training.
* **Rows are never dropped from validation:** a treatment may remove rows from the training side only, so every candidate is scored on identical, untouched validation rows.
* **Drawn once:** the 20 pairs are drawn once on `frame` (`frame_splits`) and reused by sections 1.6, 1.7 and 1.8. A treatment that drops rows does it inside a split, on its training half, so the pairs stay valid.

In [ ]:
# Twenty independent 80/20 draws that keep the class balance of the target, so every
# candidate below is judged on the same twenty validation sets
splits = StratifiedShuffleSplit(n_splits=20, test_size=0.2,
                                random_state=RANDOM_STATE)

# The 20 (training rows, validation rows) pairs, reused by every comparison below
frame_splits = list(splits.split(frame, frame["repeat_purchase_90d"]))

In [ ]:
# Check: every split has the same size and keeps the target rate of the full table
pd.DataFrame([{"training rows": len(tr), "validation rows": len(va),
               "training rate": identifier.iloc[tr].mean(), "validation rate": identifier.iloc[va].mean()}
              for tr, va in frame_splits]).describe().loc[["mean", "min", "max"]].round(3)

The **baseline** every later section has to beat: no treatment, only the default median/mode fill inside each split.

In [ ]:
baseline = held_out_f1(partial(fill_missing, plan=median_mode),
                       frame, frame_splits, numeric, categorical)

print(f"baseline F1 {baseline.f1.mean():.4f} +/- {sem(baseline.f1):.4f}"
      f"   (min {baseline.f1.min():.4f}, max {baseline.f1.max():.4f})")

**Conclusions:**
* **The splits hold:** each of the 20 splits has **5,175 training rows** and **1,294 validation rows**, and the share of repeat purchases is **0.56 in both**, the same as the full table, so the stratification works.
* **Baseline:** with no treatment and only the median/mode fill, the logistic regression reaches **F1 = 0.6803 ± 0.0032** (between 0.6454 and 0.7042 across the splits). Every treatment in sections 1.6 to 1.8 is compared against this value, on the same 20 splits.
* **Not converged:** on these unscaled columns the solver stops at its 1,000-iteration cap on every split. Scaling, compared in section 1.8, is expected to fix this.
* **Limitation:** `test.csv` covers a later period (May 2025 to October 2026) than `train` (July 2015 to February 2025). A random split mixes future purchases into training, so these F1 scores are optimistic. They are used to **compare** treatments with each other, not to estimate the score on the test set.

____________________________________________________________________________________________________

## 1.6. Outliers

### 1.6.1. Outlier Detection

### 1.6.2. Outlier Treatment

**Conclusions:**

____________________________________________________________________________________________________

## 1.7. Missing Values Comparison and Selection

### 1.7.1. Numerical Features

### 1.7.2. Categorical Features

### 1.7.3. Final Imputation

**Conclusions:**

____________________________________________________________________________________________________

## 1.8. Encoding and Scaling

### 1.8.1. Encoding

### 1.8.2. Scaling

**Conclusions:**